In [1]:
import math
import random


def sigmoid(x):
    return 1 / (1 + math.e ** (-x))


def sigmoid_derivative(out):
    # производная сигмоиды, выраженная через её же выход: s' = s * (1 - s)
    return out * (1 - out)


class Neuron:
    def __init__(self, num_inputs, activation="sigmoid"):
        # сигмоида по краям плоская: и на входе 10, и на входе 20 она выдаёт почти 1.
        # такой нейрон перестаёт реагировать на подкрутку весов, а значит и учиться —
        # сеть двигает вес и не видит, стало лучше или хуже.
        # поэтому стартовые веса держим мелкими, чтобы сумма вышла около нуля,
        # где сигмоида крутая. и чем больше входов складываем, тем мельче нужен
        # каждый вес, чтобы сумма осталась небольшой — отсюда деление на корень
        limit = 1 / (num_inputs ** 0.5)

        # веса именно случайные, а не одинаковые: иначе все нейроны слоя посчитают
        # одно и то же, получат один и тот же градиент и навсегда останутся копиями
        self.weights = [random.uniform(-limit, limit) for _ in range(num_inputs)]

        # а bias можно с нуля: непохожими нейроны уже сделали случайные веса,
        # зато sigmoid(0) = 0.5 — самая крутая точка, лучший старт для обучения
        self.bias = 0.0

        self.activation = activation

    def forward(self, inputs):
        total = 0
        for i in range(len(inputs)):
            total += inputs[i] * self.weights[i]

        total += self.bias

        self.last_inputs = inputs
        if self.activation == "sigmoid":
            self.last_output = sigmoid(total)
        else:
            self.last_output = total

        return self.last_output

    def backward(self, d_output, lr):
        # d_output — это dL/d(мой выход), пришло от следующего слоя

        # 1. проносим градиент через активацию: dL/d(total)
        if self.activation == "sigmoid":
            d_total = d_output * sigmoid_derivative(self.last_output)
        else:
            d_total = d_output

        # 2. считаем градиент по входам ДО обновления весов —
        #    предыдущий слой должен получить производную по старым весам
        d_inputs = [d_total * w for w in self.weights]

        # 3. шаг градиентного спуска: dL/d(вес_i) = d_total * вход_i
        for i in range(len(self.weights)):
            self.weights[i] -= lr * d_total * self.last_inputs[i]
        self.bias -= lr * d_total

        return d_inputs


class Layer:
    def __init__(self, num_neurons, num_inputs, activation="sigmoid"):
        self.neurons = [Neuron(num_inputs, activation) for _ in range(num_neurons)]
        self.num_inputs = num_inputs

    def forward(self, inputs):
        outputs = []
        for neuron in self.neurons:
            answer = neuron.forward(inputs)
            outputs.append(answer)
        return outputs

    def backward(self, d_outputs, lr):
        # каждый вход слоя влиял на ВСЕ нейроны сразу,
        # поэтому их вклады в градиент складываем поэлементно
        d_inputs = [0.0] * self.num_inputs
        for neuron, d_out in zip(self.neurons, d_outputs):
            contribution = neuron.backward(d_out, lr)
            for i in range(self.num_inputs):
                d_inputs[i] += contribution[i]
        return d_inputs


class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, inputs):
        data = inputs
        for layer in self.layers:
            data = layer.forward(data)
        return data

    def backward(self, d_outputs, lr):
        # то же самое, что forward, только слои идут в обратном порядке
        grad = d_outputs
        for layer in reversed(self.layers):
            grad = layer.backward(grad, lr)


In [2]:
# обучение сети предсказывать цену дома

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_percentage_error

df = pd.read_csv("../lessons/ParisHousing.csv")

X = df.drop(columns=["price", "cityCode"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# min/max считаем ТОЛЬКО по train и теми же числами масштабируем test.
# если взять min/max по всем данным — информация о тесте протечёт в обучение
x_min, x_max = X_train.min(), X_train.max()
X_train_scaled = ((X_train - x_min) / (x_max - x_min)).values.tolist()
X_test_scaled = ((X_test - x_min) / (x_max - x_min)).values.tolist()

# цену тоже сжимаем в 0..1: иначе ошибка в миллионы даёт гигантские градиенты
y_min, y_max = y_train.min(), y_train.max()
y_train_scaled = ((y_train - y_min) / (y_max - y_min)).tolist()

random.seed(42)

network = NeuralNetwork([
    Layer(num_neurons=16, num_inputs=X.shape[1], activation="sigmoid"),
    Layer(num_neurons=1, num_inputs=16, activation="linear"),  # выход без сигмоиды!
])

learning_rate = 0.05
epochs = 15

for epoch in range(1, epochs + 1):
    total_loss = 0.0

    # перемешиваем порядок строк каждую эпоху, чтобы сеть не заучивала последовательность
    order = list(range(len(X_train_scaled)))
    random.shuffle(order)

    for idx in order:
        prediction = network.forward(X_train_scaled[idx])[0]

        error = prediction - y_train_scaled[idx]
        total_loss += error ** 2

        # производная MSE = (pred - y)^2 по предсказанию равна 2 * (pred - y)
        network.backward([2 * error], learning_rate)

    print(f"эпоха {epoch:>2}   MSE (на нормированной цене) = {total_loss / len(order):.8f}")

# предсказываем на тесте и разворачиваем нормализацию обратно, в настоящие цены
predictions_scaled = [network.forward(row)[0] for row in X_test_scaled]
predictions = [p * (y_max - y_min) + y_min for p in predictions_scaled]

print()
print("R2 на тесте:", round(r2_score(y_test, predictions), 4))
print("средняя абсолютная процентная ошибка:",
      f"{round(mean_absolute_percentage_error(y_test, predictions) * 100, 2)}%")
print("пример — реальная цена / предсказание:")
for real, pred in list(zip(y_test.tolist(), predictions))[:5]:
    print(f"  ${real:>12,.0f}   ${pred:>12,.0f}")


эпоха  1   MSE (на нормированной цене) = 0.01100581
эпоха  2   MSE (на нормированной цене) = 0.00004020
эпоха  3   MSE (на нормированной цене) = 0.00003101
эпоха  4   MSE (на нормированной цене) = 0.00002545
эпоха  5   MSE (на нормированной цене) = 0.00002165
эпоха  6   MSE (на нормированной цене) = 0.00001896
эпоха  7   MSE (на нормированной цене) = 0.00001717
эпоха  8   MSE (на нормированной цене) = 0.00001572
эпоха  9   MSE (на нормированной цене) = 0.00001466
эпоха 10   MSE (на нормированной цене) = 0.00001380
эпоха 11   MSE (на нормированной цене) = 0.00001328
эпоха 12   MSE (на нормированной цене) = 0.00001273
эпоха 13   MSE (на нормированной цене) = 0.00001240
эпоха 14   MSE (на нормированной цене) = 0.00001172
эпоха 15   MSE (на нормированной цене) = 0.00001143

R2 на тесте: 0.9999
средняя абсолютная процентная ошибка: 2.52%
пример — реальная цена / предсказание:
  $   7,964,370   $   7,915,339
  $   3,713,549   $   3,698,291
  $   8,553,019   $   8,554,835
  $   6,428,666   $ 